In [20]:
pip install evaluate

In [21]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
import evaluate
from sklearn.metrics import f1_score, accuracy_score

# Setup and Tokenization
MODEL_ID = "tasksource/ModernBERT-large-nli"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [23]:
# Data Cleansing. It avoids CUDA crashing.

print("Loading and cleaning data...")
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("dev.csv")

# Droping all blank spaces of the columns
train_df = train_df.dropna(subset=['premise', 'hypothesis', 'label']).copy()
val_df = val_df.dropna(subset=['premise', 'hypothesis', 'label']).copy()

# Maping to every numerical possible option to integer
label_mapping = {
    "NOT_ENTAILMENT": 0, "ENTAILMENT": 1,
    "0": 0, "1": 1,
    "0.0": 0, "1.0": 1,
    0: 0, 1: 1,
    0.0: 0, 1.0: 1
}

train_df['label'] = train_df['label'].map(label_mapping)
val_df['label'] = val_df['label'].map(label_mapping)

#Dropping all rows that was mapped  to NaN
train_df = train_df.dropna(subset=['label'])
val_df = val_df.dropna(subset=['label'])

# Forcing being integers
train_df['label'] = train_df['label'].astype(int)
val_df['label'] = val_df['label'].astype(int)

print(f"Cleaned Training Data: {len(train_df)} rows")
print(f"Cleaned Validation Data: {len(val_df)} rows")

# Convert pandas to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

def preprocess_function(examples):
    texts = [f"{p} [SEP] {h}" for p, h in zip(examples['premise'], examples['hypothesis'])]
    return tokenizer(texts, truncation=True, max_length=256, padding="max_length")

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)


# Metric and Model Initialization
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "NOT_ENTAILMENT", 1: "ENTAILMENT"},
    label2id={"NOT_ENTAILMENT": 0, "ENTAILMENT": 1}
)


Loading and cleaning data...
Cleaned Training Data: 24432 rows
Cleaned Validation Data: 6736 rows
Tokenizing datasets...


Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: tasksource/ModernBERT-large-nli
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([2])            
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([3, 1024]) vs model:torch.Size([2, 1024])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [24]:
# Training Loop
training_args = TrainingArguments(
    output_dir="./modernbert-large-rte-final",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting the training")
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Starting the training


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.831884,0.208577,0.937500,0.939849
2,0.167114,0.402165,0.937797,0.939963
3,0.012260,0.498406,0.936758,0.939351


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4581, training_loss=0.36243505442514606, metrics={'train_runtime': 2189.3951, 'train_samples_per_second': 33.478, 'train_steps_per_second': 2.092, 'total_flos': 3.875731269142118e+16, 'train_loss': 0.36243505442514606, 'epoch': 3.0})

In [25]:

# # Threshold Tunning(Optimization)

# A. Get raw probability predictions
predictions = trainer.predict(tokenized_val)
raw_scores = predictions.predictions

# B. Convert raw logits into percentages
probabilities = torch.nn.functional.softmax(torch.tensor(raw_scores), dim=-1).numpy()

# C. Isolate Entailment (Class 1) probabilities and true labels
prob_entailment = probabilities[:, 1]
true_labels = predictions.label_ids

# Trackers
best_acc_threshold = 0.5
best_acc = 0.0

best_f1_threshold = 0.5
best_f1 = 0.0

# E. Loop through thresholds from 0.10 to 0.90
for thresh in np.arange(0.1, 0.9, 0.01):
    custom_guesses = (prob_entailment >= thresh).astype(int)

    current_acc = accuracy_score(true_labels, custom_guesses)
    current_f1 = f1_score(true_labels, custom_guesses, average="binary")

    # Best accuracy
    if current_acc > best_acc:
        best_acc = current_acc
        best_acc_threshold = thresh

    # Best F1 score
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_f1_threshold = thresh

default_guesses = (prob_entailment >= 0.5).astype(int)
print(f"Default 0.50 Threshold Accuracy: {accuracy_score(true_labels, default_guesses):.4f}")
print(f"Default 0.50 Threshold F1:       {f1_score(true_labels, default_guesses, average='binary'):.4f}")
print("----------------------------------")
print(f"Optimal ACCURACY Threshold: {best_acc_threshold:.2f} (Score: {best_acc:.4f})")
print(f"Optimal F1 Score Threshold: {best_f1_threshold:.2f} (Score: {best_f1:.4f})")

Default 0.50 Threshold Accuracy: 0.9378
Default 0.50 Threshold F1:       0.9400
----------------------------------
Optimal ACCURACY Threshold: 0.15 (Score: 0.9391)
Optimal F1 Score Threshold: 0.15 (Score: 0.9416)


In [29]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

ORIGINAL_MODEL_ID = "answerdotai/ModernBERT-large"
tokenizer_orig = AutoTokenizer.from_pretrained(ORIGINAL_MODEL_ID)

train_df = pd.read_csv("train.csv").dropna(subset=['premise', 'hypothesis', 'label']).copy()
val_df = pd.read_csv("dev.csv").dropna(subset=['premise', 'hypothesis', 'label']).copy()

label_mapping = {
    "NOT_ENTAILMENT": 0, "ENTAILMENT": 1,
    "0": 0, "1": 1, "0.0": 0, "1.0": 1,
    0: 0, 1: 1, 0.0: 0, 1.0: 1
}
train_df['label'] = train_df['label'].map(label_mapping).astype(int)
val_df['label'] = val_df['label'].map(label_mapping).astype(int)

train_dataset_orig = Dataset.from_pandas(train_df)
val_dataset_orig = Dataset.from_pandas(val_df)

def preprocess_orig(examples):
    texts = [f"{p} [SEP] {h}" for p, h in zip(examples['premise'], examples['hypothesis'])]
    return tokenizer_orig(texts, truncation=True, max_length=256, padding="max_length")

tokenized_train_orig = train_dataset_orig.map(preprocess_orig, batched=True)
tokenized_val_orig = val_dataset_orig.map(preprocess_orig, batched=True)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics_orig(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

model_orig = AutoModelForSequenceClassification.from_pretrained(
    ORIGINAL_MODEL_ID,
    num_labels=2,
    id2label={0: "NOT_ENTAILMENT", 1: "ENTAILMENT"},
    label2id={"NOT_ENTAILMENT": 0, "ENTAILMENT": 1}
)

training_args_orig = TrainingArguments(
    output_dir="./modernbert-original",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True
)

trainer_orig = Trainer(
    model=model_orig,
    args=training_args_orig,
    train_dataset=tokenized_train_orig,
    eval_dataset=tokenized_val_orig,
    processing_class=tokenizer_orig,
    compute_metrics=compute_metrics_orig
)

print("Training original ModernBERT-large...")
trainer_orig.train()

Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Training original ModernBERT-large...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.983347,0.253694,0.915529,0.917236
2,0.479512,0.418093,0.925030,0.927640
3,0.087481,0.515276,0.925475,0.928265


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4581, training_loss=0.5908952041422793, metrics={'train_runtime': 2162.8931, 'train_samples_per_second': 33.888, 'train_steps_per_second': 2.118, 'total_flos': 3.875731269142118e+16, 'train_loss': 0.5908952041422793, 'epoch': 3.0})

In [31]:
# Threshold tuning for original ModernBERT
orig_preds = trainer_orig.predict(tokenized_val_orig)
orig_probs = torch.softmax(torch.tensor(orig_preds.predictions), dim=-1).numpy()
orig_labels = orig_preds.label_ids

prob_entailment_orig = orig_probs[:, 1]

best_acc_thresh, best_acc = 0.5, 0.0
best_f1_thresh, best_f1 = 0.5, 0.0

for thresh in np.arange(0.05, 0.95, 0.01):
    preds = (prob_entailment_orig >= thresh).astype(int)
    acc = accuracy_score(orig_labels, preds)
    f1 = f1_score(orig_labels, preds, average="binary")
    if acc > best_acc:
        best_acc, best_acc_thresh = acc, thresh
    if f1 > best_f1:
        best_f1, best_f1_thresh = f1, thresh

default_preds = (prob_entailment_orig >= 0.5).astype(int)
print(f"Default 0.50 Accuracy: {accuracy_score(orig_labels, default_preds):.4f}")
print(f"Default 0.50 F1:       {f1_score(orig_labels, default_preds, average='binary'):.4f}")
print("----------------------------------")
print(f"Optimal ACCURACY Threshold: {best_acc_thresh:.2f} (Score: {best_acc:.4f})")
print(f"Optimal F1 Threshold:       {best_f1_thresh:.2f} (Score: {best_f1:.4f})")

Default 0.50 Accuracy: 0.9255
Default 0.50 F1:       0.9283
----------------------------------
Optimal ACCURACY Threshold: 0.06 (Score: 0.9267)
Optimal F1 Threshold:       0.06 (Score: 0.9299)


In [33]:
from sklearn.metrics import f1_score, accuracy_score

orig_preds = trainer_orig.predict(tokenized_val_orig)
orig_probs = torch.softmax(torch.tensor(orig_preds.predictions), dim=-1).numpy()
orig_labels = orig_preds.label_ids

print(f"Original ModernBERT standalone F1: {f1_score(orig_labels, np.argmax(orig_probs, axis=1), average='binary'):.4f}")
print(f"Tasksource ModernBERT standalone F1: {f1_score(true_labels, np.argmax(tasksource_probs, axis=1), average='binary'):.4f}")

print("\nEnsemble sweep:")
for w_orig, w_task in [(0.3, 0.7), (0.4, 0.6), (0.5, 0.5), (0.6, 0.4), (0.2, 0.8)]:
    ens = w_orig * orig_probs + w_task * tasksource_probs
    p_ent = ens[:, 1]
    best_f1, best_thresh = 0.0, 0.5
    for thresh in np.arange(0.05, 0.95, 0.01):
        f1 = f1_score(true_labels, (p_ent >= thresh).astype(int), average="binary")
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    default_f1 = f1_score(true_labels, (p_ent >= 0.5).astype(int), average="binary")
    print(f"  orig={w_orig} tasksource={w_task} | default F1: {default_f1:.4f} | best F1: {best_f1:.4f} @ thresh={best_thresh:.2f}")

Original ModernBERT standalone F1: 0.9283
Tasksource ModernBERT standalone F1: 0.9400

Ensemble sweep:
  orig=0.3 tasksource=0.7 | default F1: 0.9413 | best F1: 0.9418 @ thresh=0.56
  orig=0.4 tasksource=0.6 | default F1: 0.9420 | best F1: 0.9422 @ thresh=0.43
  orig=0.5 tasksource=0.5 | default F1: 0.9417 | best F1: 0.9417 @ thresh=0.50
  orig=0.6 tasksource=0.4 | default F1: 0.9301 | best F1: 0.9366 @ thresh=0.40
  orig=0.2 tasksource=0.8 | default F1: 0.9410 | best F1: 0.9420 @ thresh=0.24
